In [1]:
# Importando bibliotecas
# requisições HTTP
import requests

# leitura do HTML
from bs4 import BeautifulSoup

# estruturação dos dados e json
import pandas as pd
import json

# controle de tempo
import time

# tratamento de texto para URL
from urllib.parse import quote
from urllib.parse import urlparse

# Controle do Chrome
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Import para intervalo aleatório
import random

# Acessar Windows
import os

In [2]:
# Bloco 1.8: termos de busca focados em vagas remotas em sites de empresas

job_terms_scientist = [
    '"cientista de dados" '
    '("trabalhe conosco" OR carreiras OR careers OR jobs OR openings OR opportunities OR '
    '"join our team" OR "join us" OR "job opening" OR "open positions" OR '
    '"banco de talentos" OR "venha fazer parte" OR "faça parte do time") '
    '(remoto OR remote OR "100% remoto" OR "fully remote" OR "home office") '
    '-híbrido -hybrid '
    '-site:linkedin.com '
    '-site:gupy.io '
    '-site:indeed.com '
    '-site:glassdoor.com '
    '-site:catho.com.br '
    '-site:infojobs.com.br '
    '-site:vagas.com.br '
    '-site:careerjet.com.br '
    '-site:jooble.org '
    '-site:jobs.lever.co '
    '-site:boards.greenhouse.io '
    '-site:smartrecruiters.com '
    '-site:icims.com '
    '-site:myworkdayjobs.com '
    '-site:workday.com '
    '-site:ashbyhq.com '
    '-site:jobvite.com '
    '-site:recrut.ai '
    '-site:inhire.app '
    '-site:empregare.com '
    '-site:rhgestor.com.br '
    '-site:rhstorm.com.br'
]

job_terms_analyst = [    
    '"analista de dados" '
    '("trabalhe conosco" OR carreiras OR careers OR jobs OR openings OR opportunities OR '
    '"join our team" OR "join us" OR "job opening" OR "open positions" OR '
    '"banco de talentos" OR "venha fazer parte" OR "faça parte do time") '
    '(remoto OR remote OR "100% remoto" OR "fully remote" OR "home office") '
    '-híbrido -hybrid '
    '-site:linkedin.com '
    '-site:gupy.io '
    '-site:indeed.com '
    '-site:glassdoor.com '
    '-site:catho.com.br '
    '-site:infojobs.com.br '
    '-site:vagas.com.br '
    '-site:careerjet.com.br '
    '-site:jooble.org '
    '-site:jobs.lever.co '
    '-site:boards.greenhouse.io '
    '-site:smartrecruiters.com '
    '-site:icims.com '
    '-site:myworkdayjobs.com '
    '-site:workday.com '
    '-site:ashbyhq.com '
    '-site:jobvite.com '
    '-site:recrut.ai '
    '-site:inhire.app '
    '-site:empregare.com '
    '-site:rhgestor.com.br '
    '-site:rhstorm.com.br'
]

In [3]:
# Quantidade máxima de jobs
max_jobs = 75

In [4]:
# Configuração para uso com Google Chrome (Selenium)

chrome_options = Options()
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)

In [5]:
# Bloco 1.9: intervalo entre buscas

min_sleep = 8
max_sleep = 18

In [6]:
# Bloco 2: acessar uma página no Chrome

url = "https://www.google.com"

driver.get(url)

In [7]:
# Bloco 3: função para gerar URLs de busca no Google

def build_search_url(term, page=0):
    base_url = "https://www.google.com/search"
    
    query_encoded = quote(term)
    start = page * 10
    
    url = f"{base_url}?q={query_encoded}&start={start}"
    
    return url

In [8]:
# Bloco 4: abrir uma busca de teste no Google

search_term = job_terms_scientist[0]
url = build_search_url(search_term, page=0)

driver.get(url)
print(url)

https://www.google.com/search?q=%22cientista%20de%20dados%22%20%28%22trabalhe%20conosco%22%20OR%20carreiras%20OR%20careers%20OR%20jobs%20OR%20openings%20OR%20opportunities%20OR%20%22join%20our%20team%22%20OR%20%22join%20us%22%20OR%20%22job%20opening%22%20OR%20%22open%20positions%22%20OR%20%22banco%20de%20talentos%22%20OR%20%22venha%20fazer%20parte%22%20OR%20%22fa%C3%A7a%20parte%20do%20time%22%29%20%28remoto%20OR%20remote%20OR%20%22100%25%20remoto%22%20OR%20%22fully%20remote%22%20OR%20%22home%20office%22%29%20-h%C3%ADbrido%20-hybrid%20-site%3Alinkedin.com%20-site%3Agupy.io%20-site%3Aindeed.com%20-site%3Aglassdoor.com%20-site%3Acatho.com.br%20-site%3Ainfojobs.com.br%20-site%3Avagas.com.br%20-site%3Acareerjet.com.br%20-site%3Ajooble.org%20-site%3Ajobs.lever.co%20-site%3Aboards.greenhouse.io%20-site%3Asmartrecruiters.com%20-site%3Aicims.com%20-site%3Amyworkdayjobs.com%20-site%3Aworkday.com%20-site%3Aashbyhq.com%20-site%3Ajobvite.com%20-site%3Arecrut.ai%20-site%3Ainhire.app%20-site%3Aempreg

In [9]:
url = build_search_url(search_term, page=0)
driver.get(url)

input("Se o Google pedir verificação, resolva no navegador e pressione Enter para continuar...")

Se o Google pedir verificação, resolva no navegador e pressione Enter para continuar... 


''

In [10]:
# ==========================================
# Bloco 5: coletar links relevantes dos resultados no Google
# ==========================================
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

def normalize_url(url):
    try:
        parsed = urlparse(url.strip())

        # remove fragmentos tipo #:~:text=
        fragment = ""

        # remove query params de tracking
        blocked_params = {
            "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
            "gclid", "fbclid", "igshid", "ref", "refs"
        }
        clean_query = urlencode([
            (k, v) for k, v in parse_qsl(parsed.query, keep_blank_values=True)
            if k.lower() not in blocked_params
        ])

        # remove barra final do path
        clean_path = parsed.path.rstrip("/")

        normalized = urlunparse((
            parsed.scheme.lower(),
            parsed.netloc.lower().replace("www.", ""),
            clean_path,
            parsed.params,
            clean_query,
            fragment
        ))

        return normalized
    except:
        return url


def is_job_like_url(url):
    url_lower = url.lower()

    positive_signals = [
        "/jobs/",
        "/job/",
        "/careers/job",
        "/careers/jobs",
        "/vaga/",
        "/vagas/",
        "/oportunidades/",
        "/opportunities/",
        "/position/",
        "/positions/",
        "/career/job",
        "jobdescription",
        "job-descriptions",
        "jobs/view",
        "jobid",
        "reqid",
        "requisition",
        "opportunity",
        "apply",
        "detalhes-da-vaga",
        "job-details",
        "viewjob",
        "trabalhe-conosco"
    ]

    negative_signals = [
        "/search",
        "/busca",
        "/tag/",
        "/category/",
        "/categorias/",
        "/blog/",
        "/noticias/",
        "/news/",
        "/artigos/",
        "/article/",
        "/insights/",
        "/wiki/",
        "/curso/",
        "/cursos/",
        "/formacao/",
        "/carreiras/",
        "/careers/",
        "/vagas"  # página genérica sem detalhe
    ]

    # forte sinal de vaga
    has_positive = any(signal in url_lower for signal in positive_signals)

    # sinais de páginas genéricas
    has_negative = any(signal in url_lower for signal in negative_signals)

    # exceção: se tiver /vagas/ e mais níveis depois, pode ser vaga específica
    if "/vagas/" in url_lower:
        path = urlparse(url_lower).path.strip("/")
        if len(path.split("/")) >= 2:
            has_positive = True

    # exceção: se /careers/ aparecer junto com /job/ ou /jobs/, pode aceitar
    if "/careers/" in url_lower and ("/job/" in url_lower or "/jobs/" in url_lower):
        has_positive = True
        has_negative = False

    return has_positive and not has_negative


def get_job_links(driver):
    links = []

    blocked_domains = [
        "linkedin.com",
        "gupy.io",
        "indeed.com",
        "glassdoor.com",
        "catho.com.br",
        "infojobs.com.br",
        "vagas.com.br",
        "careerjet.com.br",
        "jooble.org",
        "jobs.lever.co",
        "boards.greenhouse.io",
        "smartrecruiters.com",
        "icims.com",
        "myworkdayjobs.com",
        "workday.com",
        "ashbyhq.com",
        "jobvite.com",
        "recrut.ai",
        "inhire.app",
        "empregare.com",
        "rhgestor.com.br",
        "rhstorm.com.br"
    ]

    anchors = driver.find_elements("tag name", "a")

    for a in anchors:
        href = a.get_attribute("href")

        if not href:
            continue

        href = href.strip()

        if not href.startswith("http"):
            continue

        href_norm = normalize_url(href)
        domain = urlparse(href_norm).netloc.lower().replace("www.", "")

        if any(blocked in domain for blocked in blocked_domains):
            continue

        if "google.com" in domain:
            continue

        if not is_job_like_url(href_norm):
            continue

        links.append(href_norm)

    # remove duplicados preservando ordem
    links = list(dict.fromkeys(links))

    return links

In [11]:
# ==========================================
# Bloco 6: coletar links com controle de progresso por termo
# ==========================================

def is_google_blocked(driver):
    page_source = driver.page_source.lower()
    current_url = driver.current_url.lower()
    title = driver.title.lower()

    sinais_bloqueio = [
        "unusual traffic",
        "sistemas detectaram tráfego incomum",
        "detected unusual traffic",
        "our systems have detected unusual traffic",
        "sorry",
        "captcha",
        "/sorry/",
        "recaptcha",
        "não sou um robô",
        "i'm not a robot"
    ]

    for sinal in sinais_bloqueio:
        if sinal in page_source or sinal in current_url or sinal in title:
            return True

    try:
        resultados = driver.find_elements(By.CSS_SELECTOR, "a")
        if len(resultados) < 5 and "google" in current_url:
            return True
    except:
        pass

    return False


def load_progress_data(arquivo_progresso):
    if not os.path.exists(arquivo_progresso):
        return {}

    try:
        with open(arquivo_progresso, "r", encoding="utf-8") as f:
            conteudo = f.read().strip()

        if not conteudo:
            return {}

        data = json.loads(conteudo)

        if isinstance(data, dict):
            return data
        else:
            return {}

    except (json.JSONDecodeError, OSError):
        return {}


def save_progress_data(arquivo_progresso, progress_data):
    with open(arquivo_progresso, "w", encoding="utf-8") as f:
        json.dump(progress_data, f, ensure_ascii=False, indent=2)


all_jobs = []

arquivo_existente = "vagas_encontradas.xlsx"
arquivo_progresso = "progresso_busca_google.json"

# histórico de links já salvos
if os.path.exists(arquivo_existente):
    df_existente = pd.read_excel(arquivo_existente)

    if "link" in df_existente.columns:
        visited_links = set(
            df_existente["link"]
            .dropna()
            .astype(str)
            .str.strip()
            .apply(normalize_url)
            .tolist()
        )
    else:
        visited_links = set()

    print(f"Links já existentes no arquivo: {len(visited_links)}")
else:
    visited_links = set()
    print("Arquivo de vagas anteriores não encontrado. Busca iniciará do zero.")

# progresso por termo
progress_data = load_progress_data(arquivo_progresso)

search_groups = [
    ("cientista de dados", job_terms_scientist),
    ("analista de dados", job_terms_analyst),
]

google_block_detected = False

for role_name, term_list in search_groups:
    print(f"\nIniciando buscas para: {role_name}")

    for term in term_list:
        term_key = f"{role_name}||{term}"
        start_page = progress_data.get(term_key, 0)

        print(f"\nTermo: {term}")
        print(f"Iniciando da página: {start_page}")

        page = start_page
        consecutive_pages_without_new_links = 0

        while len(all_jobs) < max_jobs:
            search_url = build_search_url(term, page)
            print(f"Buscando página {page}: {search_url}")

            driver.get(search_url)
            time.sleep(random.uniform(8, 15))

            if is_google_blocked(driver):
                print("\nGoogle bloqueou ou suspeitou da navegação automatizada.")
                print("Busca interrompida para evitar novas tentativas seguidas.")
                google_block_detected = True
                break

            new_links = get_job_links(driver)

            if not new_links:
                print("Nenhum link relevante encontrado nesta página.")
                consecutive_pages_without_new_links += 1
            else:
                added_in_page = 0

                for link in new_links:
                    link_norm = normalize_url(link)

                    if link_norm not in visited_links:
                        visited_links.add(link_norm)
                        all_jobs.append({
                            "cargo": role_name,
                            "link": link_norm
                        })
                        added_in_page += 1

                        if len(all_jobs) >= max_jobs:
                            break

                print(f"Links relevantes encontrados na página: {len(new_links)}")
                print(f"Novos links adicionados nesta página: {added_in_page}")
                print(f"Total acumulado na execução: {len(all_jobs)}")

                if added_in_page == 0:
                    consecutive_pages_without_new_links += 1
                    print("Página sem novidade útil.")
                else:
                    consecutive_pages_without_new_links = 0

            # salva progresso: próxima página a visitar no futuro
            progress_data[term_key] = page + 1
            save_progress_data(arquivo_progresso, progress_data)

            if consecutive_pages_without_new_links >= 2:
                print("Duas páginas seguidas sem novidade útil. Encerrando este termo.")
                break

            page += 1
            time.sleep(random.uniform(10, 18))

        if google_block_detected or len(all_jobs) >= max_jobs:
            break

    if google_block_detected or len(all_jobs) >= max_jobs:
        break

print(f"\nTotal final de vagas coletadas nesta execução: {len(all_jobs)}")

if google_block_detected:
    print("Coleta finalizada por bloqueio do Google.")

Links já existentes no arquivo: 58

Iniciando buscas para: cientista de dados

Termo: "cientista de dados" ("trabalhe conosco" OR carreiras OR careers OR jobs OR openings OR opportunities OR "join our team" OR "join us" OR "job opening" OR "open positions" OR "banco de talentos" OR "venha fazer parte" OR "faça parte do time") (remoto OR remote OR "100% remoto" OR "fully remote" OR "home office") -híbrido -hybrid -site:linkedin.com -site:gupy.io -site:indeed.com -site:glassdoor.com -site:catho.com.br -site:infojobs.com.br -site:vagas.com.br -site:careerjet.com.br -site:jooble.org -site:jobs.lever.co -site:boards.greenhouse.io -site:smartrecruiters.com -site:icims.com -site:myworkdayjobs.com -site:workday.com -site:ashbyhq.com -site:jobvite.com -site:recrut.ai -site:inhire.app -site:empregare.com -site:rhgestor.com.br -site:rhstorm.com.br
Iniciando da página: 17
Buscando página 17: https://www.google.com/search?q=%22cientista%20de%20dados%22%20%28%22trabalhe%20conosco%22%20OR%20carreiras

In [12]:
# ==========================================
# BLOCO 7 - LISTAGEM FINAL DAS VAGAS
# ==========================================

output_file = "vagas_encontradas.xlsx"

# transformar vagas novas em dataframe
df_vagas = pd.DataFrame(all_jobs).copy()

# garantir colunas esperadas
expected_cols = ["cargo", "link"]
for col in expected_cols:
    if col not in df_vagas.columns:
        df_vagas[col] = None

df_vagas = df_vagas[expected_cols].copy()

# remover linhas sem link
df_vagas = df_vagas.dropna(subset=["link"])

# padronizar
df_vagas["cargo"] = df_vagas["cargo"].astype(str).str.strip()
df_vagas["link"] = df_vagas["link"].astype(str).str.strip().apply(normalize_url)

# manter só links válidos
df_vagas = df_vagas[
    (df_vagas["link"] != "") &
    (df_vagas["link"].str.startswith("http"))
].copy()

# remover duplicados das vagas novas
df_vagas = df_vagas.drop_duplicates(subset=["link"]).reset_index(drop=True)

qtd_novas_execucao = len(df_vagas)

# se já existir histórico, juntar e remover duplicados
if os.path.exists(output_file):
    df_antigas = pd.read_excel(output_file)

    for col in expected_cols:
        if col not in df_antigas.columns:
            df_antigas[col] = None

    df_antigas = df_antigas[expected_cols].copy()
    df_antigas = df_antigas.dropna(subset=["link"])
    df_antigas["cargo"] = df_antigas["cargo"].astype(str).str.strip()
    df_antigas["link"] = df_antigas["link"].astype(str).str.strip().apply(normalize_url)

    total_antes = len(df_antigas)

    df_vagas = pd.concat([df_antigas, df_vagas], ignore_index=True)
    df_vagas = df_vagas.drop_duplicates(subset=["link"], keep="first").reset_index(drop=True)
else:
    total_antes = 0

total_depois = len(df_vagas)
qtd_realmente_adicionada = total_depois - total_antes

# criar source a partir do domínio
df_vagas["source"] = df_vagas["link"].apply(
    lambda x: urlparse(x).netloc.replace("www.", "")
)

# criar coluna title vazia por enquanto
df_vagas["title"] = None

# reorganizar colunas
df_vagas = df_vagas[["cargo", "title", "link", "source"]]

# ordenar
df_vagas = df_vagas.sort_values(
    by=["cargo", "source", "link"],
    ascending=[True, True, True]
).reset_index(drop=True)

# salvar em xlsx
df_vagas.to_excel(output_file, index=False)

print(f"Total de vagas no arquivo: {len(df_vagas)}")
print(f"Novas vagas coletadas nesta execução: {qtd_novas_execucao}")
print(f"Novas vagas realmente adicionadas ao histórico: {qtd_realmente_adicionada}")
print(f"Arquivo salvo em: {output_file}")

if 'google_block_detected' in globals() and google_block_detected:
    print("A coleta foi interrompida por possível bloqueio do Google.")

if 'arquivo_progresso' in globals():
    print(f"Progresso de busca mantido em: {arquivo_progresso}")

display(df_vagas.head(5))

Total de vagas no arquivo: 75
Novas vagas coletadas nesta execução: 17
Novas vagas realmente adicionadas ao histórico: 17
Arquivo salvo em: vagas_encontradas.xlsx
A coleta foi interrompida por possível bloqueio do Google.
Progresso de busca mantido em: progresso_busca_google.json


,cargo,title,link,source
0,analista de dados,None,https://alura.com.br/carreiras/analise-de-dado...,alura.com.br
1,analista de dados,None,https://apibr.com/ui/vagas,apibr.com
2,analista de dados,None,https://careers.cbre.com/sv_SE/careers/JobDeta...,careers.cbre.com
3,analista de dados,None,https://careers.dhl.com/amer/pt/dhl-campinas,careers.dhl.com
4,analista de dados,None,https://careers.shopee.com.br/jobs,careers.shopee.com.br
